In [1]:
# ============================================================
import pandas as pd
import numpy as np
import logging
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# 导入自定义模块
from src.core.database import DatabaseManager
from src.core.data_fetcher import DataFetcher
from src.core.spread_calculator import SpreadCalculator, create_standard_crack_spreads
from src.core.indicators import IndicatorBuilder
from src.core.feature_engineering import FeatureEngineer
from src.core.ml_models import MLModel, SignalGenerator
from src.core.backtest import BacktestEngine, PerformanceAnalyzer
from src.core.visualization import Visualizer

# ============================================================
# 配置日志
# ============================================================
log_dir = Path('logs')
log_dir.mkdir(exist_ok=True)
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(log_dir / 'debug_strategy.log', encoding='utf-8'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

logger.info("="*60)
logger.info("开始运行调试版本策略")
logger.info("="*60)

# ============================================================
# 全局变量初始化
# ============================================================
print("\n[1/9] 初始化模块...")

# 数据库和工具类
db_path = "data/trading_data.db"
db = DatabaseManager(db_path)
fetcher = DataFetcher()
spread_calc = SpreadCalculator(db)
indicator_builder = IndicatorBuilder(db)
feature_engineer = FeatureEngineer(indicator_builder)
visualizer = Visualizer(output_dir="outputs/Crack_Spread_Results")

# 数据存储容器
price_data = {}          # 存储各品种价格数据
spread_data = {}         # 存储价差数据
macro_data = {}          # 存储宏观数据
fundamental_data = {}    # 存储基本面数据

# 特征和模型
features_df = None       # 合并后的特征DataFrame
X_train = None          # 训练集特征
X_test = None           # 测试集特征
y_train = None          # 训练集标签
y_test = None           # 测试集标签
train_idx = None        # 训练集索引
test_idx = None         # 测试集索引
model = None            # 训练好的模型
selected_features = []  # 选择的特征列表

# 回测结果
signals = None          # 交易信号
equity_curve = None     # 权益曲线
trade_log = None        # 交易日志
performance_report = None  # 绩效报告

print("✓ 模块初始化完成")

START_DATE = "2000-01-01"

INFO:__main__:============================================================
INFO:__main__:开始运行调试版本策略
INFO:__main__:============================================================
INFO:src.core.database:数据表创建完成
INFO:src.core.database:数据库已初始化: data/trading_data.db



[1/9] 初始化模块...
✓ 模块初始化完成


In [2]:
# # ============================================================
# # 步骤1：数据获取
# # ============================================================

# print("\n[2/9] 开始数据获取...")
# logger.info("\n" + "="*60)
# logger.info("步骤1：数据获取")
# logger.info("="*60)

# # 期货品种配置
# symbols_config = {
#     'CL': 'CL=F',    # WTI原油
#     'RBOB': 'RB=F',  # RBOB汽油
#     'HO': 'HO=F'     # 取暖油/柴油
# }

# # 获取期货数据
# for name, symbol in symbols_config.items():
#     print(f"  获取 {name} ({symbol}) 数据...")
#     logger.info(f"获取 {name} 数据...")
    
#     df = fetcher.fetch_yfinance_data(
#         symbol, 
#         start_date=START_DATE,
#         interval='1d'
#     )
    
#     if not df.empty:
#         # 期货复权处理
#         # df_adjusted = FuturesAdjuster.adjust_futures_roll(df, method='ratio')
#         df_adjusted = df.copy()  # 简化处理，直接使用原始数据
#         # 保存到数据库
#         db.insert_price_data(df_adjusted, name, 'adjusted')
#         price_data[name] = df_adjusted
        
#         print(f"  ✓ {name}: {len(df_adjusted)} 条记录")
#         logger.info(f"{name} 数据获取成功: {len(df_adjusted)} 条记录")
#     else:
#         print(f"  ✗ {name} 数据获取失败")
#         logger.warning(f"{name} 数据获取失败")
# ============================================================
# 步骤1：从数据库提取数据
# ============================================================

print("\n[2/9] 开始从数据库提取数据...")
logger.info("\n" + "="*60)
logger.info("步骤1：数据提取")
logger.info("="*60)

# 定义数据起始日期
START_DATE = "2000-01-01"

# 1. 提取期货价格数据
print("  [1.1] 提取期货价格数据...")
symbols = ['CL', 'RBOB', 'HO']

for symbol in symbols:
    try:
        df = db.get_price_data(
            symbol=symbol,
            start_date=START_DATE,
            columns=['date', 'open', 'high', 'low', 'close', 'volume']
        )
        
        if not df.empty:
            # 设置索引
            if 'date' in df.columns:
                df.set_index('date', inplace=True)
            
            # 确保索引无时区（统一处理）
            if hasattr(df.index, 'tz') and df.index.tz is not None:
                df.index = df.index.tz_localize(None)
            
            price_data[symbol] = df
            print(f"    ✓ {symbol}: {len(df)} 条记录")
            logger.info(f"{symbol} 数据提取成功: {len(df)} 条记录")
        else:
            print(f"    ✗ {symbol}: 数据库中无数据")
            logger.warning(f"{symbol} 数据库中无数据，请先运行数据更新脚本")
            
    except Exception as e:
        print(f"    ✗ {symbol}: 提取失败 - {e}")
        logger.error(f"{symbol} 提取失败: {e}")

INFO:__main__:
INFO:__main__:步骤1：数据提取
INFO:__main__:============================================================



[2/9] 开始从数据库提取数据...
  [1.1] 提取期货价格数据...


INFO:__main__:CL 数据提取成功: 6326 条记录


    ✓ CL: 6326 条记录


INFO:__main__:RBOB 数据提取成功: 6281 条记录


    ✓ RBOB: 6281 条记录


INFO:__main__:HO 数据提取成功: 6320 条记录


    ✓ HO: 6320 条记录


In [3]:
# # 获取宏观数据
# print("\n  获取宏观经济数据...")
# logger.info("获取宏观经济数据...")

# # VIX波动率指数
# vix_data = fetcher.fetch_index_data('VIX', start_date=START_DATE)
# if not vix_data.empty:
#     db.insert_price_data(vix_data, 'VIX', 'index')
#     macro_data['VIX'] = vix_data
#     print(f"  ✓ VIX: {len(vix_data)} 条记录")
#     logger.info(f"VIX数据获取成功: {len(vix_data)} 条记录")

# # 美元指数
# dxy_data = fetcher.fetch_index_data('DXY', start_date=START_DATE)
# if not dxy_data.empty:
#     db.insert_price_data(dxy_data, 'DXY', 'index')
#     macro_data['DXY'] = dxy_data
#     print(f"  ✓ DXY: {len(dxy_data)} 条记录")
#     logger.info(f"DXY数据获取成功: {len(dxy_data)} 条记录")


# # 获取基本面数据
# print("\n  获取基本面数据...")
# logger.info("获取基本面数据...")
# eia_data = fetcher.fetch_eia_data('PET.WCRSTUS1.W')
# if not eia_data.empty:
#     for date, row in eia_data.iterrows():
#         db.insert_fundamental_data(
#             'EIA',
#             date.strftime('%Y-%m-%d'),
#             date.strftime('%Y-%m-%d'),
#             row.to_dict()
#         )
#     fundamental_data['EIA'] = eia_data
#     print(f"  ✓ EIA: {len(eia_data)} 条记录")
#     logger.info(f"EIA数据获取成功: {len(eia_data)} 条记录")


# print("✓ 数据获取完成")
# logger.info("数据获取完成\n")
# 2. 提取宏观数据
print("\n  [1.2] 提取宏观数据...")
macro_symbols = ['VIX', 'DXY']

for symbol in macro_symbols:
    try:
        df = db.get_price_data(
            symbol=symbol,
            start_date=START_DATE,
            columns=['date', 'close']
        )
        
        if not df.empty:
            # 设置索引
            if 'date' in df.columns:
                df.set_index('date', inplace=True)
            
            # 确保索引无时区
            # if hasattr(df.index, 'tz') and df.index.tz is not None:
            #     df.index = df.index.tz_localize(None)
            
            macro_data[symbol] = df
            print(f"    ✓ {symbol}: {len(df)} 条记录")
            logger.info(f"{symbol} 数据提取成功: {len(df)} 条记录")
        else:
            print(f"    ✗ {symbol}: 数据库中无数据")
            logger.warning(f"{symbol} 数据库中无数据")
            
    except Exception as e:
        print(f"    ✗ {symbol}: 提取失败 - {e}")
        logger.error(f"{symbol} 提取失败: {e}")

# 3. 提取基本面数据
print("\n  [1.3] 提取基本面数据...")
try:
    eia_df = db.get_fundamental_data(
        data_source='EIA',
        start_date=START_DATE
    )
    
    if not eia_df.empty:
        # 解析JSON数据并转换为DataFrame
        import json
        
        # 提取所有数据字段
        data_list = []
        for _, row in eia_df.iterrows():
            data_dict = json.loads(row['data_json'])
            data_dict['report_date'] = row['report_date']
            data_list.append(data_dict)
        
        fundamental_data['EIA'] = pd.DataFrame(data_list)
        fundamental_data['EIA']['report_date'] = pd.to_datetime(
            fundamental_data['EIA']['report_date']
        )
        fundamental_data['EIA'].set_index('report_date', inplace=True)
        
        print(f"    ✓ EIA: {len(fundamental_data['EIA'])} 条记录")
        logger.info(f"EIA 数据提取成功: {len(fundamental_data['EIA'])} 条记录")
    else:
        print("    ✗ EIA: 数据库中无数据")
        logger.warning("EIA 数据库中无数据")
        
except Exception as e:
    print(f"    ✗ EIA: 提取失败 - {e}")
    logger.error(f"EIA 提取失败: {e}")

# 4. 数据验证
print("\n  [1.4] 数据验证...")
print("    期货数据:")
for symbol, df in price_data.items():
    date_range = f"{df.index.min()} 至 {df.index.max()}"
    print(f"      {symbol}: {len(df)} 条, 日期范围: {date_range}")

print("    宏观数据:")
for symbol, df in macro_data.items():
    date_range = f"{df.index.min()} 至 {df.index.max()}"
    print(f"      {symbol}: {len(df)} 条, 日期范围: {date_range}")

if 'EIA' in fundamental_data:
    eia = fundamental_data['EIA']
    date_range = f"{eia.index.min()} 至 {eia.index.max()}"
    print(f"    基本面数据:")
    print(f"      EIA: {len(eia)} 条, 日期范围: {date_range}")

# 5. 检查数据完整性
print("\n  [1.5] 数据完整性检查...")
required_symbols = ['CL', 'RBOB', 'HO']
missing_symbols = [s for s in required_symbols if s not in price_data or price_data[s].empty]

if missing_symbols:
    error_msg = f"缺少必要的期货数据: {missing_symbols}"
    print(f"    ❌ {error_msg}")
    logger.error(error_msg)
    print("\n    💡 解决方案:")
    print("    1. 运行数据更新脚本: python scripts/daily_data_update.py")
    print("    2. 检查网络连接和数据源可用性")
    print("    3. 查看日志文件: logs/daily_update.log")
    raise ValueError(error_msg)
else:
    print("    ✅ 所有必要数据已就绪")

print("\n✓ 数据提取完成")
logger.info("数据提取完成\n")



# 🔍 调试点1：在此处设置断点，检查 price_data, macro_data 的内容
# 可以在调试控制台输入: price_data.keys(), len(price_data['CL'])


  [1.2] 提取宏观数据...


INFO:__main__:VIX 数据提取成功: 6498 条记录


    ✓ VIX: 6498 条记录


INFO:__main__:DXY 数据提取成功: 6527 条记录
INFO:__main__:EIA 数据提取成功: 209 条记录
INFO:__main__:数据提取完成



    ✓ DXY: 6527 条记录

  [1.3] 提取基本面数据...
    ✓ EIA: 209 条记录

  [1.4] 数据验证...
    期货数据:
      CL: 6326 条, 日期范围: 2000-08-23 00:00:00-04:00 至 2025-10-31 00:00:00-04:00
      RBOB: 6281 条, 日期范围: 2000-11-01 00:00:00-05:00 至 2025-10-31 00:00:00-04:00
      HO: 6320 条, 日期范围: 2000-09-01 00:00:00-04:00 至 2025-10-31 00:00:00-04:00
    宏观数据:
      VIX: 6498 条, 日期范围: 2000-01-03 00:00:00-06:00 至 2025-10-31 00:00:00-05:00
      DXY: 6527 条, 日期范围: 2000-01-03 00:00:00-05:00 至 2025-10-31 00:00:00-04:00
    基本面数据:
      EIA: 209 条, 日期范围: 2020-01-05 00:00:00 至 2023-12-31 00:00:00

  [1.5] 数据完整性检查...
    ✅ 所有必要数据已就绪

✓ 数据提取完成


In [4]:
# ============================================================
# 步骤2：计算价差
# ============================================================
print("\n[3/9] 开始计算价差...")
logger.info("\n" + "="*60)
logger.info("步骤2：计算价差")
logger.info("="*60)


# 创建标准裂解价差配置

# print(price_data["CL"].head(20))
create_standard_crack_spreads(spread_calc)

# 计算3:2:1裂解价差
if all(symbol in price_data for symbol in ['CL', 'RBOB', 'HO']):
    print("  计算 CRACK_3_2_1 价差...")
    
    spread_df = spread_calc.calculate_spread(
        'CRACK_3_2_1',
        price_data,
        price_column='close'
    )
    
    # 添加统计特征
    spread_df = spread_calc.get_spread_statistics(spread_df, window=20)
    spread_data['CRACK_3_2_1'] = spread_df
    
    # 保存到数据库
    db.insert_indicator_data(
        'CRACK_3_2_1',
        spread_df[['spread']],
        metadata={'type': 'crack_spread', 'ratio': '3:2:1'}
    )
    
    print(f"  ✓ 价差数据点: {len(spread_df)}")
    logger.info(f"3:2:1裂解价差计算完成: {len(spread_df)} 个数据点")
    
    # 平稳性检验
    print("  进行ADF平稳性检验...")
    indicator_builder.test_stationarity(
        spread_df['spread'],
        name='CRACK_3_2_1 Spread'
    )
else:
    print("  ✗ 缺少必要的价格数据")
    logger.error("缺少计算价差所需的价格数据")

print("✓ 价差计算完成")
logger.info("价差计算完成\n")

# 🔍 调试点2：在此处设置断点，检查 spread_data['CRACK_3_2_1'] 的统计特征
# print(spread_data['CRACK_3_2_1'].head(20))

INFO:__main__:
INFO:__main__:步骤2：计算价差
INFO:__main__:============================================================
INFO:src.core.spread_calculator:价差配置已保存到数据库: RBOB_CL_1_1
INFO:src.core.spread_calculator:创建价差配置: RBOB_CL_1_1 - 多头(1.0xRBOB) - 空头(1.0xCL)
INFO:src.core.spread_calculator:价差配置已保存到数据库: HO_CL_1_1
INFO:src.core.spread_calculator:创建价差配置: HO_CL_1_1 - 多头(1.0xHO) - 空头(1.0xCL)
INFO:src.core.spread_calculator:价差配置已保存到数据库: CRACK_3_2_1
INFO:src.core.spread_calculator:创建价差配置: CRACK_3_2_1 - 多头(2.0xRBOB + 1.0xHO) - 空头(3.0xCL)
INFO:src.core.spread_calculator:价差配置已保存到数据库: CRACK_2_1_1
INFO:src.core.spread_calculator:创建价差配置: CRACK_2_1_1 - 多头(1.0xRBOB + 1.0xHO) - 空头(2.0xCL)
INFO:src.core.spread_calculator:价差配置已保存到数据库: CRACK_5_3_2
INFO:src.core.spread_calculator:创建价差配置: CRACK_5_3_2 - 多头(3.0xRBOB + 2.0xHO) - 空头(5.0xCL)
INFO:src.core.spread_calculator:标准裂解价差配置创建完成
INFO:src.core.spread_calculator:RBOB: 价格从美元/加仑转换为美元/桶 (乘以42)
INFO:src.core.spread_calculator:HO: 价格从美元/加仑转换为美元/桶 (乘以42)
INFO:src.core.sp


[3/9] 开始计算价差...
  计算 CRACK_3_2_1 价差...


INFO:src.core.spread_calculator:价差统计特征计算完成，窗口: 20
INFO:src.core.database:插入了 0 条新指标数据
INFO:__main__:3:2:1裂解价差计算完成: 6276 个数据点
INFO:src.core.indicators:


  ✓ 价差数据点: 6276
  进行ADF平稳性检验...


INFO:src.core.indicators:ADF平稳性检验结果 - CRACK_3_2_1 Spread
INFO:src.core.indicators:==================================================
INFO:src.core.indicators:ADF统计量: -3.777377
INFO:src.core.indicators:P值: 0.003147
INFO:src.core.indicators:使用滞后阶数: 8
INFO:src.core.indicators:观测值数量: 6267
INFO:src.core.indicators:临界值:
INFO:src.core.indicators:  1%: -3.431394
INFO:src.core.indicators:  5%: -2.862001
INFO:src.core.indicators:  10%: -2.567016
INFO:src.core.indicators:结论: CRACK_3_2_1 Spread 是平稳序列 (p < 0.05)
INFO:src.core.indicators:==================================================

INFO:__main__:价差计算完成



✓ 价差计算完成


### 📊 价格单位转换说明

在计算裂解价差时，不同产品的价格单位不同：
- **WTI原油 (CL)**: 美元/桶
- **RBOB汽油 (RBOB)**: 美元/加仑
- **取暖油/柴油 (HO)**: 美元/加仑

为了正确计算价差，需要将所有价格**统一转换为美元/桶**：
- 1桶 = 42加仑
- RBOB和HO的价格需要乘以42

**转换示例**：
```python
# RBOB汽油: 2.50 美元/加仑 → 105.00 美元/桶 (2.50 × 42)
# WTI原油:  70.00 美元/桶 → 70.00 美元/桶 (不变)
```

`SpreadCalculator.calculate_spread()` 方法会自动进行单位转换。

In [5]:
# ============================================================
# 步骤3：特征工程
# ============================================================
print("\n[4/9] 开始特征工程...")
logger.info("\n" + "="*60)
logger.info("步骤3：特征工程")
logger.info("="*60)

# 获取主价差数据
main_spread = spread_data.get('CRACK_3_2_1')
if main_spread is None or main_spread.empty:
    print("  ✗ 价差数据不可用，停止执行")
    logger.error("价差数据不可用")
    raise ValueError("价差数据不可用")

# 1. 创建价差特征
print("  [3.1] 创建价差特征...")
spread_features = feature_engineer.create_spread_features(main_spread)
print(f"    ✓ 价差特征: {len(spread_features.columns)} 个")

# 2. 创建价格特征
print("  [3.2] 创建价格特征...")
price_features = feature_engineer.create_price_features(price_data)
print(f"    ✓ 价格特征: {len(price_features.columns)} 个")

# 3. 创建技术指标特征
print("  [3.3] 创建技术指标特征...")
technical_features = pd.DataFrame(index=main_spread.index)
for symbol, df in price_data.items():
    if symbol in ['CL', 'RBOB', 'HO']:
        tech_df = feature_engineer.create_technical_features(df, symbol)
        # 选择关键列
        key_cols = [col for col in tech_df.columns 
                   if any(x in col for x in ['RSI', 'MACD', 'BB_percent'])]
        if key_cols:
            technical_features = technical_features.join(tech_df[key_cols], how='outer')
print(f"    ✓ 技术指标特征: {len(technical_features.columns)} 个")

# # 4. 创建季节性特征
# print("  [3.4] 创建季节性特征...")
# seasonal_features = feature_engineer.create_seasonal_features(
#     pd.DataFrame(index=main_spread.index)
# )
# print(f"    ✓ 季节性特征: {len(seasonal_features.columns)} 个")

# 5. 创建宏观特征（使用merge_asof对齐时间戳）
print("  [3.5] 创建宏观特征...")
# 创建基准DataFrame - 使用reset_index()确保正确的datetime类型
# 关键：直接赋值 macro_features['_timestamp'] = macro_features.index 会导致类型变为object
temp_macro = main_spread.reset_index()
temp_macro.columns = ['date'] + list(main_spread.columns)

# 确保date列是datetime类型且无时区
temp_macro['date'] = pd.to_datetime(temp_macro['date'],utc=True)
if hasattr(temp_macro['date'].dtype, 'tz') and temp_macro['date'].dtype.tz is not None:
    temp_macro['date'] = temp_macro['date'].dt.tz_localize(None)

print(f"主数据时间类型: {temp_macro['date'].dtype}, 前5行:")
print(temp_macro[['date']].head())

for symbol in ['VIX', 'DXY']:
    df = db.get_price_data(symbol)
    if not df.empty and 'close' in df.columns:
        # 准备宏观数据：重置索引，计算收益率
        macro_df = df[['close']].copy()
        macro_df[f'{symbol}_return_1d'] = macro_df['close'].pct_change()
        macro_df = macro_df.reset_index()
        macro_df.columns = ['date', f'{symbol}_close', f'{symbol}_return_1d']
        
        # 确保有干净的datetime列（不带时区）
        # 如果原数据带时区，先用utc=True转换，再移除时区
        macro_df['date'] = pd.to_datetime(macro_df['date'], utc=True)
        if hasattr(macro_df['date'].dtype, 'tz') and macro_df['date'].dtype.tz is not None:
            macro_df['date'] = macro_df['date'].dt.tz_localize(None)
        
        print(f"{symbol}数据时间类型: {macro_df['date'].dtype}")
        
        # 使用merge_asof进行时间对齐（向后填充）
        temp_macro = pd.merge_asof(
            temp_macro.sort_values('date'),
            macro_df[['date', f'{symbol}_close', f'{symbol}_return_1d']].sort_values('date'),
            on='date',
            direction='backward'  # 使用最近的历史数据
        )
        
        # 显示对齐效果
        aligned_count = temp_macro[f'{symbol}_close'].notna().sum()
        missing_count = temp_macro[f'{symbol}_close'].isnull().sum()
        print(f"✓ {symbol}: {len(df)}条原始 → {aligned_count}条对齐（缺失{missing_count}个）")

# 恢复为索引格式（确保索引无时区）
macro_features = temp_macro.set_index('date')
# 再次确认索引无时区
# if hasattr(macro_features.index.dtype, 'tz') and macro_features.index.tz is not None:
#     macro_features.index = macro_features.index.tz_localize(None)

# 只保留宏观数据列，去除来自main_spread的列（避免与spread_features重复）
macro_cols = [col for col in macro_features.columns if any(x in col for x in ['VIX', 'DXY'])]
macro_features = macro_features[macro_cols]

print("\n宏观特征前20行:")
print(macro_features.head(20))
print(f"宏观特征索引类型: {macro_features.index.dtype}")

print(f"    ✓ 宏观特征: {len(macro_features.columns)} 个（已时间对齐）")

# 6. 创建目标变量
print("  [3.6] 创建目标变量...")
target_df = feature_engineer.create_target_variable(
    main_spread,
    method='sharpe_regress',
    forward_period=10,
    threshold=0.05
)
print(f"    ✓ 目标变量创建完成")

# 🔍 调试点3：在此处设置断点，检查各个特征DataFrame
# 可以查看: spread_features.head(), price_features.shape, technical_features.columns

# ============================================================
# 合并特征
# ============================================================
print("\n  [3.7] 合并所有特征...")
logger.info("合并特征...")

all_features_list = [
    spread_features,
    price_features,
    technical_features,
    # seasonal_features,
    macro_features,
    target_df[['target']]
]
# print(pd.concat(all_features_list, axis=1).head(20))

# 诊断：打印合并前的状态
print("\n  诊断信息 - 合并前各DataFrame状态:")

df_names = ['价差特征', '价格特征', '技术指标', '季节性特征', '宏观特征', '目标变量']

# 统一清理所有DataFrame的索引时区
print("\n  [3.7.1] 统一清理索引时区...")
for i, (name, df) in enumerate(zip(df_names, all_features_list)):
    # 检查索引是否有时区
    df.index = pd.to_datetime(df.index, utc=True)
    if hasattr(df.index, 'tz') and df.index.tz is not None:
        print(f"    {name}: 移除时区 {df.index.tz}")
        all_features_list[i].index = df.index.tz_localize(None)
    
    # 打印诊断信息
    null_count = df.isnull().sum().sum()
    index_type = type(df.index).__name__
    index_dtype = df.index.dtype if hasattr(df.index, 'dtype') else 'N/A'
    print(f"    {name}: {len(df)}样本, {len(df.columns)}列, {null_count}缺失值, 索引类型:{index_type}({index_dtype})")
    if null_count > 0:
        null_cols = df.isnull().sum()
        null_cols = null_cols[null_cols > 0]
        print(f"      缺失值列: {dict(list(null_cols.items())[:3])}")

# 合并特征
print("\n  [3.7.2] 开始合并特征...")
features_df = feature_engineer.merge_all_features(all_features_list)

print(f"\n  合并后状态:")
print(f"    总样本数: {len(features_df)}")
print(f"    总特征数: {len(features_df.columns)}")
print(f"    总缺失值: {features_df.isnull().sum().sum()}")

# ============================================================
# 清理缺失值
# ============================================================
print("\n  [3.8] 清理缺失值...")

# 显示缺失值最多的列
total_nulls = features_df.isnull().sum().sum()
if total_nulls > 0:
    null_counts = features_df.isnull().sum()
    cols_with_nulls = null_counts[null_counts > 0].sort_values(ascending=False)
    print(f"    缺失值最多的前5列:")
    for col, count in cols_with_nulls.head(5).items():
        pct = count / len(features_df) * 100
        print(f"      {col}: {count} ({pct:.2f}%)")

# 步骤1：前向填充
print("\n    步骤1: ffill前向填充...")
exclude_target = features_df.columns.difference(['target', 'forward_return'])
features_df[exclude_target] = features_df[exclude_target].fillna(method='ffill')
remaining_nulls = features_df.isnull().sum().sum()
print(f"      剩余缺失值: {remaining_nulls}")

# # 步骤2：后向填充
# if remaining_nulls > 0:
#     print("    步骤2: bfill后向填充...")
#     features_df = features_df.fillna(method='bfill')
#     remaining_nulls = features_df.isnull().sum().sum()
#     print(f"      剩余缺失值: {remaining_nulls}")

# # 步骤3：均值填充
# if remaining_nulls > 0:
#     print("    步骤3: 均值填充...")
#     numeric_cols = features_df.select_dtypes(include=[np.number]).columns
#     features_df[numeric_cols] = features_df[numeric_cols].fillna(
#         features_df[numeric_cols].mean()
#     )
#     remaining_nulls = features_df.isnull().sum().sum()
#     print(f"      剩余缺失值: {remaining_nulls}")

# 步骤4：删除仍有缺失值的列
# if remaining_nulls > 0:
#     null_cols = features_df.columns[features_df.isnull().any()].tolist()
#     print(f"    步骤4: 删除 {len(null_cols)} 个仍有缺失值的列")
#     print(f"      删除的列: {null_cols[:5]}")
#     features_df = features_df.dropna(axis=1)

print(f"\n  最终清理结果:")
print(f"    剩余样本数: {len(features_df)}")
print(f"    剩余特征数: {len(features_df.columns) - 2}")  # 减去target和forward_return
print(f"    缺失值: {features_df.isnull().sum().sum()}")

print("✓ 特征工程完成")
logger.info(f"特征构建完成，总特征数: {len(features_df.columns) - 2}")
logger.info(f"样本数: {len(features_df)}")

# 🔍 调试点4：在此处设置断点，检查 features_df
# 可以使用: features_df.describe(), features_df.head(), features_df.isnull().sum()

INFO:__main__:
INFO:__main__:步骤3：特征工程
INFO:__main__:============================================================
INFO:src.core.feature_engineering:创建价差特征完成，特征数: 39
INFO:src.core.feature_engineering:创建价格特征完成，特征数: 48



[4/9] 开始特征工程...
  [3.1] 创建价差特征...
    ✓ 价差特征: 51 个
  [3.2] 创建价格特征...


INFO:src.core.indicators:计算RSI完成，周期: 14
INFO:src.core.indicators:计算MACD完成，参数: 12/26/9


    ✓ 价格特征: 48 个
  [3.3] 创建技术指标特征...


INFO:src.core.indicators:计算布林带完成，窗口: 20, 标准差: 2.0
INFO:src.core.feature_engineering:创建技术指标特征完成: CL_
INFO:src.core.indicators:计算RSI完成，周期: 14
INFO:src.core.indicators:计算MACD完成，参数: 12/26/9
INFO:src.core.indicators:计算布林带完成，窗口: 20, 标准差: 2.0
INFO:src.core.feature_engineering:创建技术指标特征完成: RBOB_
INFO:src.core.indicators:计算RSI完成，周期: 14
INFO:src.core.indicators:计算MACD完成，参数: 12/26/9
INFO:src.core.indicators:计算布林带完成，窗口: 20, 标准差: 2.0
INFO:src.core.feature_engineering:创建技术指标特征完成: HO_


    ✓ 技术指标特征: 15 个
  [3.5] 创建宏观特征...
主数据时间类型: datetime64[ns], 前5行:
                 date
0 2000-11-01 05:00:00
1 2000-11-02 05:00:00
2 2000-11-03 05:00:00
3 2000-11-06 05:00:00
4 2000-11-07 05:00:00
VIX数据时间类型: datetime64[ns]
✓ VIX: 6498条原始 → 6276条对齐（缺失0个）


INFO:src.core.feature_engineering:创建夏普比率回归目标变量
INFO:__main__:合并特征...
INFO:src.core.feature_engineering:特征合并完成，总特征数: 119, 样本数: 6276


DXY数据时间类型: datetime64[ns]
✓ DXY: 6527条原始 → 6276条对齐（缺失0个）

宏观特征前20行:
                     VIX_close  VIX_return_1d   DXY_close  DXY_return_1d
date                                                                    
2000-11-01 05:00:00  23.629999      -0.071877  115.559998      -0.009344
2000-11-02 05:00:00  24.280001       0.027507  115.610001       0.000433
2000-11-03 05:00:00  23.920000      -0.014827  114.970001      -0.005536
2000-11-06 05:00:00  23.670000      -0.010452  115.690002       0.006263
2000-11-07 05:00:00  24.520000       0.035910  115.519997      -0.001469
2000-11-08 05:00:00  24.910000       0.015905  116.230003       0.006146
2000-11-09 05:00:00  25.660000       0.030108  115.339996      -0.007657
2000-11-10 05:00:00  27.200001       0.060016  115.790001       0.003902
2000-11-13 05:00:00  28.530001       0.048897  115.699997      -0.000777
2000-11-14 05:00:00  29.059999       0.018577  116.220001       0.004494
2000-11-15 05:00:00  26.809999      -0.077426  116.34999

INFO:__main__:特征构建完成，总特征数: 117
INFO:__main__:样本数: 6276


      剩余缺失值: 1254

  最终清理结果:
    剩余样本数: 6276
    剩余特征数: 117
    缺失值: 1254
✓ 特征工程完成


In [6]:
features_df.tail(10)

,spread,spread_name,spread_mean,spread_std,spread_min,spread_max,spread_zscore,spread_percentile,spread_pct_change,spread_pct_change_5d,...,HO_RSI_14,HO_MACD,HO_MACD_signal,HO_MACD_hist,HO_BB_percent,VIX_close,VIX_return_1d,DXY_close,DXY_return_1d,target
date,,,,,,,,,,,,,,,,,,,,,
2025-10-20 04:00:00,73.244997,CRACK_3_2_1,71.636369,3.056616,67.566595,76.578605,0.526277,65.0,-0.000843,0.033124,...,33.213782,-0.038326,-0.027572,-0.010754,0.251562,20.780001,-0.178981,98.589996,0.001625,NaN
2025-10-21 04:00:00,72.508801,CRACK_3_2_1,71.488289,2.930223,67.566595,76.578605,0.348271,65.0,-0.010051,0.038784,...,38.026431,-0.036743,-0.029407,-0.007336,0.312813,18.230000,-0.122714,98.930000,0.003449,NaN
2025-10-22 04:00:00,75.643198,CRACK_3_2_1,71.547869,3.004812,67.566595,76.578605,1.362924,85.0,0.043228,0.071112,...,50.788717,-0.031590,-0.029843,-0.001746,0.464160,17.870001,-0.019748,98.900002,-0.000303,NaN
2025-10-23 04:00:00,77.415605,CRACK_3_2_1,71.660760,3.184609,67.566595,77.415605,1.807081,100.0,0.023431,0.102005,...,65.640842,-0.014955,-0.026866,0.011910,0.976565,18.600000,0.040851,98.940002,0.000404,NaN
2025-10-24 04:00:00,77.937005,CRACK_3_2_1,71.757000,3.347929,67.566595,77.937005,1.845919,100.0,0.006735,0.063162,...,65.123819,-0.001744,-0.021841,0.020097,1.000405,17.299999,-0.069893,98.949997,0.000101,NaN
2025-10-27 04:00:00,79.699798,CRACK_3_2_1,71.931210,3.667538,67.566595,79.699798,2.118203,100.0,0.022618,0.088126,...,65.903165,0.011259,-0.015221,0.026480,1.044393,16.370001,-0.053757,98.779999,-0.001718,NaN
2025-10-28 04:00:00,81.529199,CRACK_3_2_1,72.178740,4.134982,67.566595,81.529199,2.261306,100.0,0.022954,0.124404,...,58.564037,0.017417,-0.008693,0.026111,0.867076,15.790000,-0.035431,98.690002,-0.000911,NaN
2025-10-29 04:00:00,86.171403,CRACK_3_2_1,72.999540,5.136614,67.566595,86.171403,2.564308,100.0,0.056939,0.139182,...,62.286705,0.025003,-0.001954,0.026957,0.924162,16.420000,0.039899,99.220001,0.005370,NaN
2025-10-30 04:00:00,89.895610,CRACK_3_2_1,74.080771,6.245201,67.566595,89.895610,2.532319,100.0,0.043219,0.161208,...,73.415180,0.033510,0.005139,0.028371,0.948161,16.920000,0.030451,99.529999,0.003124,NaN


In [7]:
features_df = features_df.dropna()

In [8]:
import importlib
import src.core.ml_models
importlib.reload(src.core.ml_models)
from src.core.ml_models import MLModel,SignalGenerator

In [9]:
# ============================================================
# 步骤4：模型训练
# ============================================================
print("\n[5/9] 开始模型训练...")
logger.info("\n" + "="*60)
logger.info("步骤4：模型训练")
logger.info("="*60)

if features_df is None or features_df.empty:
    print("  ✗ 特征数据不可用，停止执行")
    logger.error("特征数据不可用")
    raise ValueError("特征数据不可用")

# 特征选择
print("  [4.1] 特征选择...")
selected_features = feature_engineer.select_features(
    features_df,
    target_col='target',
    method='variance',
    top_k=50
)
print(f"    ✓ 选择了 {len(selected_features)} 个特征")

# 创建模型
print("  [4.2] 创建模型...")
model = MLModel(model_type='gradient_boosting', task='regression')
print("    ✓ 使用 Gradient Boosting 分类器")

# 准备数据
print("  [4.3] 准备训练/测试数据...")
X_train, X_test, y_train, y_test, train_idx, test_idx = model.prepare_data(
    features_df,
    target_col='target',
    feature_cols=selected_features,
    test_size=0.15,
    scale=True
)
print(f"    ✓ 训练集: {len(X_train)} 样本")
print(f"    ✓ 测试集: {len(X_test)} 样本")

# 训练模型
print("  [4.4] 训练模型...")
model_params = {
    'n_estimators': 500,
    'max_depth': 8,
    'learning_rate': 0.1,
    'random_state': 42
}
model.train(X_train, y_train, **model_params)
print("    ✓ 模型训练完成")

# 评估模型
print("  [4.5] 评估模型...")
metrics = model.evaluate(X_test, y_test)
print(f"    ✓ 准确率: {metrics.get('accuracy', 0):.4f}")
print(f"    ✓ F1分数: {metrics.get('f1', 0):.4f}")

# 可视化特征重要性
if model.feature_importance is not None:
    print("  [4.6] 生成特征重要性图...")
    visualizer.plot_feature_importance(model.feature_importance, top_n=15)
    print("    ✓ 特征重要性图已保存")

# 保存模型
print("  [4.7] 保存模型...")
model_path = Path('models/crack_spread_model.pkl')
model_path.parent.mkdir(exist_ok=True)
model.save_model(str(model_path))
print(f"    ✓ 模型已保存到: {model_path}")

print("✓ 模型训练完成")
logger.info("模型训练完成\n")

# 🔍 调试点5：在此处设置断点，检查模型和训练结果
# 可以查看: model.feature_importance, metrics, X_train.shape, X_test.shape

# ============================================================
# 步骤5：回测
# ============================================================
print("\n[6/9] 开始回测...")
logger.info("\n" + "="*60)
logger.info("步骤5：回测")
logger.info("="*60)





INFO:__main__:
INFO:__main__:步骤4：模型训练
INFO:__main__:============================================================
INFO:src.core.feature_engineering:特征选择完成，选择了 50 个特征



[5/9] 开始模型训练...
  [4.1] 特征选择...


INFO:src.core.ml_models:数据准备完成:
INFO:src.core.ml_models:  特征数: 50
INFO:src.core.ml_models:  训练集样本数: 5275
INFO:src.core.ml_models:  测试集样本数: 932
INFO:src.core.ml_models:开始训练 gradient_boosting 模型...


    ✓ 选择了 50 个特征
  [4.2] 创建模型...
    ✓ 使用 Gradient Boosting 分类器
  [4.3] 准备训练/测试数据...
    ✓ 训练集: 5275 样本
    ✓ 测试集: 932 样本
  [4.4] 训练模型...


INFO:src.core.ml_models:Top 10 重要特征:
INFO:src.core.ml_models:              feature  importance
0     CL_volume_ma_20    0.042012
24           CL_ma_50    0.040907
1   RBOB_volume_ma_20    0.039259
2     HO_volume_ma_20    0.037614
36      spread_std_60    0.036604
37     spread_kurt_60    0.036029
17       spread_ma_60    0.035224
23           CL_ma_20    0.035156
47   spread_zscore_60    0.033353
25      spread_min_60    0.031709
INFO:src.core.ml_models:模型训练完成
INFO:src.core.ml_models:
模型评估结果:
INFO:src.core.ml_models:MSE: 7.657832
INFO:src.core.ml_models:RMSE: 2.767279
INFO:src.core.ml_models:MAE: 2.264215
INFO:src.core.ml_models:R²: -0.4465


    ✓ 模型训练完成
  [4.5] 评估模型...
    ✓ 准确率: 0.0000
    ✓ F1分数: 0.0000
  [4.6] 生成特征重要性图...


INFO:src.core.visualization:特征重要性图表已保存: outputs/Crack_Spread_Results/feature_importance.png
INFO:src.core.ml_models:模型已保存: models\crack_spread_model.pkl
INFO:__main__:模型训练完成

INFO:__main__:
INFO:__main__:步骤5：回测
INFO:__main__:============================================================


    ✓ 特征重要性图已保存
  [4.7] 保存模型...
    ✓ 模型已保存到: models\crack_spread_model.pkl
✓ 模型训练完成

[6/9] 开始回测...


In [10]:
model.predict(X_test)

array([-1.44785475e+00, -6.05255070e-01, -1.59263636e+00, -6.22092823e-01,
       -1.93772925e+00, -8.01934648e-01, -1.15515130e+00, -2.30319267e+00,
       -1.09514911e+00, -2.06462741e-01, -3.79248609e-01, -9.59792639e-01,
       -1.35394044e+00, -9.95469656e-01, -1.31472375e+00, -8.32841998e-01,
       -1.25375857e+00, -2.90712114e+00, -2.73014987e+00, -3.67631386e+00,
       -5.77305153e+00, -4.18036327e+00, -6.34190467e+00, -2.56985167e+00,
       -1.13045557e+00, -2.25680150e+00, -2.52697505e+00, -1.51009004e+00,
       -2.49007448e+00, -4.22851141e+00, -3.41705357e+00, -1.78131443e+00,
       -2.42078191e+00, -3.84300926e+00, -4.14189740e+00, -3.32670248e+00,
       -1.38854526e-01, -4.35666243e-01, -9.48543829e-01, -1.31000599e+00,
       -2.80670052e-01, -6.40400761e-01, -6.99103444e-01, -1.02567973e+00,
       -1.05934724e+00, -1.35818061e+00, -8.45241418e-01, -2.15143596e+00,
       -2.70878836e+00, -3.54281070e+00, -3.03912496e+00, -3.39835191e+00,
       -3.08574323e+00, -

In [22]:
# 生成信号
print("  [5.1] 生成交易信号...")
signal_generator = SignalGenerator(model, 
                                use_rolling_quantile=True,      # ✅ 使用滚动分位数
                                rolling_window=250,              # 250天窗口
                                upper_quantile=0.9,            # 90%分位数
                                lower_quantile=0.1,            # 10%分位数
                                signal_holding_days=10           # 信号维持5天
                                   )
signals = signal_generator.generate_signals(X_test, use_probability=True)
signals.index = test_idx
print(f"    ✓ 生成 {len(signals)} 个信号")
print(f"    信号分布: {signals.value_counts().to_dict()}")

# 获取价差价格数据
print("  [5.2] 准备价格数据...")
# 确保spread_data索引与test_idx时区一致
spread_df_for_backtest = spread_data['CRACK_3_2_1'].copy()
spread_df_for_backtest.index = pd.to_datetime(spread_df_for_backtest.index,utc=True)
if hasattr(spread_df_for_backtest.index, 'tz') and spread_df_for_backtest.index.tz is not None:
    spread_df_for_backtest.index = spread_df_for_backtest.index.tz_localize(None)

spread_prices = spread_df_for_backtest.loc[test_idx, ['spread']].copy()
spread_prices.columns = ['close']
spread_prices['volatility'] = spread_prices['close'].pct_change().rolling(20).std()
print(f"    ✓ 价格数据: {len(spread_prices)} 条")


# 运行回测
print("  [5.3] 运行回测...")
backtest_engine = BacktestEngine(
    initial_capital=1000000,
    commission_rate=0.0005,
    slippage_rate=0.0001,
    max_position=100000,
    max_capital_usage=0.1
)

equity_curve = backtest_engine.run_backtest(
    spread_prices,
    signals,
    price_col='close',
    volatility_col='volatility'
)
print(f"    ✓ 回测完成，最终权益: ${equity_curve['equity'].iloc[-1]:,.2f}")

# 获取交易日志
trade_log = backtest_engine.get_trade_log()
print(f"    ✓ 总交易次数: {len(trade_log)}")

# 绩效分析
print("  [5.4] 绩效分析...")
analyzer = PerformanceAnalyzer(
    equity_curve,
    initial_capital=1000000,
    risk_free_rate=0.02
)

performance_report = analyzer.generate_performance_report(trade_log)
print(f"    ✓ 总收益率: {performance_report.get('total_return', 0)*100:.2f}%")
print(f"    ✓ 夏普比率: {performance_report.get('sharpe_ratio', 0):.2f}")
print(f"    ✓ 最大回撤: {performance_report.get('max_drawdown', 0)*100:.2f}%")

print("✓ 回测完成")
logger.info("回测完成\n")

# 🔍 调试点6：在此处设置断点，检查回测结果
# 可以查看: equity_curve.tail(), trade_log.head(), performance_report

# ============================================================
# 步骤6：可视化
# ============================================================
print("\n[7/9] 开始可视化...")
logger.info("\n" + "="*60)
logger.info("步骤6：结果可视化")
logger.info("="*60)

print("  [6.1] 生成价格和价差图...")
visualizer.plot_price_and_spread(
    price_data,
    spread_data['CRACK_3_2_1'],
    title='Crack Spread 3:2:1'
)
print("    ✓ price_spread_chart.png")

print("  [6.2] 生成权益曲线图...")
visualizer.plot_equity_curve(equity_curve)
print("    ✓ equity_curve.png")

print("  [6.3] 生成收益率分布图...")
returns = equity_curve['equity'].pct_change().dropna()
visualizer.plot_returns_distribution(returns)
print("    ✓ returns_distribution.png")

print("  [6.4] 生成月度收益热力图...")
visualizer.plot_monthly_returns_heatmap(equity_curve)
print("    ✓ monthly_returns_heatmap.png")

print("  [6.5] 生成滚动指标图...")
visualizer.plot_rolling_metrics(equity_curve, window=60)
print("    ✓ rolling_metrics.png")

print("  [6.6] 生成交易分析图...")
visualizer.plot_trade_analysis(trade_log)
print("    ✓ trade_analysis.png")

print("✓ 可视化完成")
logger.info("可视化完成\n")

# ============================================================
# 完成
# ============================================================
print("\n" + "="*60)
print("策略执行完成！")
print("="*60)
print("\n生成的文件:")
print("  📁 data/trading_data.db          - 数据库")
print("  📁 models/crack_spread_model.pkl - 模型文件")
print("  📁 outputs/charts/*.png          - 图表文件")
print("  📁 logs/debug_strategy.log       - 日志文件")

print("\n可用的全局变量（用于调试）:")
print("  数据相关:")
print("    - price_data       : 期货价格数据字典")
print("    - spread_data      : 价差数据字典")
print("    - macro_data       : 宏观数据字典")
print("    - fundamental_data : 基本面数据字典")
print("\n  特征相关:")
print("    - spread_features  : 价差特征DataFrame")
print("    - price_features   : 价格特征DataFrame")
print("    - technical_features: 技术指标特征DataFrame")
print("    - seasonal_features: 季节性特征DataFrame")
print("    - macro_features   : 宏观特征DataFrame")
print("    - target_df        : 目标变量DataFrame")
print("    - features_df      : 合并后的完整特征DataFrame")
print("\n  模型相关:")
print("    - model            : 训练好的模型")
print("    - X_train, X_test  : 训练/测试特征")
print("    - y_train, y_test  : 训练/测试标签")
print("    - selected_features: 选择的特征列表")
print("\n  回测相关:")
print("    - signals          : 交易信号Series")
print("    - equity_curve     : 权益曲线DataFrame")
print("    - trade_log        : 交易日志DataFrame")
print("    - performance_report: 绩效报告字典")

print("\n💡 调试提示:")
print("  1. 在VS Code中打开此文件")
print("  2. 点击行号左侧设置断点（蓝点）")
print("  3. 按F5或点击'运行和调试'启动调试")
print("  4. 在'变量'面板查看所有变量的值")
print("  5. 在'调试控制台'输入变量名查看详细信息")
print("  例如: price_data.keys(), features_df.shape, model.feature_importance")

logger.info("\n" + "="*60)
logger.info("所有任务完成！")
logger.info("="*60)

# 🔍 最终调试点：程序结束前，所有变量都已计算完成
# 现在可以检查任何变量的最终状态

INFO:src.core.ml_models:使用滚动分位数模式: window=250, 上分位数=0.9, 下分位数=0.1
INFO:src.core.ml_models:滚动分位数统计:
INFO:src.core.ml_models:  上阈值范围: [0.1097, 0.9985], 均值: 0.5759
INFO:src.core.ml_models:  下阈值范围: [-5.0795, -1.5294], 均值: -3.0417
INFO:src.core.ml_models:回归信号统计:
INFO:src.core.ml_models:  预测值范围: [-9.5793, 3.0636]
INFO:src.core.ml_models:  做多信号(1): 105 (11.3%)
INFO:src.core.ml_models:  观望信号(0): 742 (79.6%)
INFO:src.core.ml_models:  做空信号(-1): 85 (9.1%)
INFO:src.core.ml_models:生成交易信号完成，信号分布:
INFO:src.core.ml_models: 0    742
 1    105
-1     85
Name: signal, dtype: int64


  [5.1] 生成交易信号...


INFO:src.core.ml_models:应用10天信号维持后，信号分布:
INFO:src.core.ml_models: 0    398
 1    295
-1    239
Name: signal, dtype: int64
INFO:src.core.backtest:开始运行回测...
INFO:src.core.backtest:初始资金: $1,000,000.00
INFO:src.core.backtest:手续费率: 0.050%
INFO:src.core.backtest:滑点率: 0.010%
INFO:src.core.backtest:杠杆倍数: 10.0x
INFO:src.core.backtest:保证金比例: 10.0%
INFO:src.core.backtest:最大资金使用率: 10%
INFO:src.core.backtest:回测完成，共执行 556 笔交易
INFO:src.core.backtest:最终权益: $2,187,318.01
INFO:src.core.backtest:
INFO:src.core.backtest:绩效分析报告
INFO:src.core.backtest:============================================================
INFO:src.core.backtest:总收益率: 118.73%
INFO:src.core.backtest:年化收益率: 24.14%
INFO:src.core.backtest:年化波动率: 41.00%
INFO:src.core.backtest:夏普比率: 0.5400
INFO:src.core.backtest:索提诺比率: 0.7151
INFO:src.core.backtest:卡玛比率: 0.7769
INFO:src.core.backtest:最大回撤: -31.07%
INFO:src.core.backtest:VaR (95%): -3.34%
INFO:src.core.backtest:CVaR (95%): -5.36%
INFO:src.core.backtest:胜率: 48.30%
INFO:src.core.backtest:盈亏比: 1

    ✓ 生成 932 个信号
    信号分布: {(-9.579294184083587, 0.28189124892939477, -5.079466098582057, -1, 0.8393076219010446): 1, (-0.3533807406547973, 0.6870477670394954, -2.6301592601573494, 1, 0.0): 1, (-0.3794884997541652, 0.6870477670394954, -1.5293532180088796, 0, 0.0): 1, (-0.3792486093420331, 0.28189124892939477, -5.079466098582057, 0, 0.0): 1, (-0.37306102171412014, 0.6343328683149543, -1.574288266360118, -1, 0.0): 1, (-0.37295935528537677, 0.6870477670394954, -2.3031763481081335, 0, 0.0): 1, (-0.3659872277123976, 0.6870477670394954, -2.8869294651401045, 1, 0.0): 1, (-0.3638887545527088, 0.7747967149930662, -2.0894456790223686, -1, 0.0): 1, (-0.3617917431521986, 0.10973662011522786, -2.966779933293476, 0, 0.0): 1, (-0.360657887313367, 0.49898794292388887, -2.1539006487273125, -1, 0.0): 1, (-0.3595096008623464, 0.28189124892939477, -5.079466098582057, 1, 0.0): 1, (-0.3589449011858663, 0.4663908479792194, -3.071404885977394, 0, 0.0): 1, (-0.35791861404727077, 0.49898794292388887, -2.1539006

INFO:src.core.visualization:价格和价差图表已保存: outputs/Crack_Spread_Results/price_spread_chart.png


...

    ✓ monthly_returns_heatmap.png
  [6.5] 生成滚动指标图...


INFO:src.core.visualization:滚动指标图表已保存: outputs/Crack_Spread_Results/rolling_metrics.png


    ✓ rolling_metrics.png
  [6.6] 生成交易分析图...


INFO:src.core.visualization:交易分析图表已保存: outputs/Crack_Spread_Results/trade_analysis.png
INFO:__main__:可视化完成

INFO:__main__:
INFO:__main__:所有任务完成！
INFO:__main__:============================================================


    ✓ trade_analysis.png
✓ 可视化完成

策略执行完成！

生成的文件:
  📁 data/trading_data.db          - 数据库
  📁 models/crack_spread_model.pkl - 模型文件
  📁 outputs/charts/*.png          - 图表文件
  📁 logs/debug_strategy.log       - 日志文件

可用的全局变量（用于调试）:
  数据相关:
    - price_data       : 期货价格数据字典
    - spread_data      : 价差数据字典
    - macro_data       : 宏观数据字典
    - fundamental_data : 基本面数据字典

  特征相关:
    - spread_features  : 价差特征DataFrame
    - price_features   : 价格特征DataFrame
    - technical_features: 技术指标特征DataFrame
    - seasonal_features: 季节性特征DataFrame
    - macro_features   : 宏观特征DataFrame
    - target_df        : 目标变量DataFrame
    - features_df      : 合并后的完整特征DataFrame

  模型相关:
    - model            : 训练好的模型
    - X_train, X_test  : 训练/测试特征
    - y_train, y_test  : 训练/测试标签
    - selected_features: 选择的特征列表

  回测相关:
    - signals          : 交易信号Series
    - equity_curve     : 权益曲线DataFrame
    - trade_log        : 交易日志DataFrame
    - performance_report: 绩效报告字典

💡 调试提示:
  1. 在VS Code中打开此文件
  2. 点击行号左侧设置断点（蓝

In [27]:
# ============================================================
# 步骤7：保存结果
# ============================================================
print("\n[8/9] 保存策略结果...")
logger.info("\n" + "="*60)
logger.info("步骤7：保存结果")
logger.info("="*60)

# 重新导入模块以获取最新代码
import importlib
import src.core.result_saver
importlib.reload(src.core.result_saver)
from src.core.result_saver import ResultSaver
from datetime import datetime
from pathlib import Path

# 创建结果保存器
result_saver = ResultSaver(output_dir="outputs/Crack_Spread_Results/strategy_runs")

# 1. 保存策略配置
print("  [7.1] 保存策略配置...")

# 提取模型参数
model_params = {
    "model_type": "gradient_boosting",
    "task": model.task,
    "n_estimators": model.model.n_estimators if hasattr(model.model, 'n_estimators') else None,
    "max_depth": model.model.max_depth if hasattr(model.model, 'max_depth') else None,
    "learning_rate": model.model.learning_rate if hasattr(model.model, 'learning_rate') else None,
    "random_state": 42,
    "scaler_used": model.scaler is not None,
    "feature_count": len(selected_features)
}

# 提取信号生成参数
signal_params = {
    "use_rolling_quantile": signal_generator.use_rolling_quantile,
    "rolling_window": signal_generator.rolling_window,
    "upper_quantile": signal_generator.upper_quantile,
    "lower_quantile": signal_generator.lower_quantile,
    "signal_holding_days": signal_generator.signal_holding_days,
    "use_probability": True
}

# 提取回测参数
backtest_params = {
    "initial_capital": backtest_engine.initial_capital,
    "commission_rate": backtest_engine.commission_rate,
    "slippage_rate": backtest_engine.slippage_rate,
    "max_position": backtest_engine.max_position,
    "max_capital_usage": backtest_engine.max_capital_usage
}

# 数据信息
data_info = {
    "start_date": START_DATE,
    "end_date": datetime.now().strftime('%Y-%m-%d'),
    "symbols": list(price_data.keys()),
    "spread_type": "CRACK_3_2_1",
    "train_samples": len(X_train),
    "test_samples": len(X_test),
    "train_period": f"{train_idx.min()} to {train_idx.max()}",
    "test_period": f"{test_idx.min()} to {test_idx.max()}",
    "total_features": len(selected_features),
    "feature_categories": {
        "spread_features": len([f for f in selected_features if 'spread' in f.lower()]),
        "price_features": len([f for f in selected_features if any(s in f for s in ['CL_', 'RBOB_', 'HO_'])]),
        "technical_features": len([f for f in selected_features if any(s in f for s in ['RSI', 'MACD', 'BB'])]),
        "macro_features": len([f for f in selected_features if any(s in f for s in ['VIX', 'DXY'])])
    }
}

config_file = result_saver.save_strategy_config(
    model_params=model_params,
    signal_params=signal_params,
    backtest_params=backtest_params,
    data_info=data_info
)
print(f"    ✓ 策略配置已保存")

# 2. 保存绩效报告
print("  [7.2] 保存绩效报告...")
perf_files = result_saver.save_performance_report(
    performance_report=performance_report,
    trade_log=trade_log,
    equity_curve=equity_curve
)
print(f"    ✓ 绩效报告已保存:")
for file_type, filepath in perf_files.items():
    print(f"      - {file_type}: {Path(filepath).name}")

# 3. 保存特征信息
print("  [7.3] 保存特征信息...")
feature_file = result_saver.save_feature_info(
    selected_features=selected_features,
    feature_importance=model.feature_importance
)
print(f"    ✓ 特征信息已保存")

# 4. 保存模型指标
print("  [7.4] 保存模型指标...")
train_metrics = model.evaluate(X_train, y_train)
test_metrics = metrics  # 使用之前计算的测试集指标
metrics_file = result_saver.save_model_metrics(
    train_metrics=train_metrics,
    test_metrics=test_metrics
)
print(f"    ✓ 模型指标已保存")
print(f"      训练集准确率: {train_metrics.get('accuracy', 0):.4f}")
print(f"      测试集准确率: {test_metrics.get('accuracy', 0):.4f}")

# 5. 保存交易信号
print("  [7.5] 保存交易信号...")
# 获取概率（如果有）
try:
    if hasattr(model.model, 'predict_proba'):
        probabilities = pd.DataFrame(
            model.model.predict_proba(X_test),
            index=test_idx,
            columns=[f'prob_class_{i}' for i in range(len(model.model.classes_))]
        )
    else:
        probabilities = None
except Exception as e:
    logger.warning(f"无法获取预测概率: {e}")
    probabilities = None

signals_file = result_saver.save_signals(
    signals=signals,
    probabilities=probabilities
)
print(f"    ✓ 交易信号已保存")

# 6. 创建README
print("  [7.6] 创建README...")
result_saver.create_readme()
print(f"    ✓ README已创建")

# 显示输出目录
output_dir = result_saver.get_run_directory()
print(f"\n✓ 所有结果已保存到: {output_dir}")
logger.info(f"所有结果已保存到: {output_dir}\n")

# 打印目录结构
print("\n📂 生成的文件:")
for file in sorted(Path(output_dir).glob('*')):
    size_kb = file.stat().st_size / 1024
    print(f"  📄 {file.name:<35} ({size_kb:>8.2f} KB)")

print("\n" + "="*80)
print("💾 结果保存完成！")
print("="*80)

print("\n📊 快速查看:")
print(f"  绩效汇总: {output_dir}/performance_summary.txt")
print(f"  交易记录: {output_dir}/performance_trades.csv")
print(f"  策略配置: {output_dir}/strategy_config.json")

print("\n💡 提示:")
print("  - 所有CSV文件可以用Excel直接打开")
print("  - JSON文件包含完整的配置和指标信息")
print("  - 不同运行的结果通过时间戳文件夹区分")
print("  - 可以对比不同参数设置的效果")


INFO:__main__:
INFO:__main__:步骤7：保存结果
INFO:__main__:============================================================
INFO:__main__:步骤7：保存结果
INFO:__main__:============================================================
INFO:src.core.result_saver:结果保存器初始化完成，输出目录: outputs\Crack_Spread_Results\strategy_runs\20251102_180742
INFO:src.core.result_saver:策略配置已保存: outputs\Crack_Spread_Results\strategy_runs\20251102_180742\strategy_config.json
INFO:src.core.result_saver:绩效报告JSON已保存: outputs\Crack_Spread_Results\strategy_runs\20251102_180742\performance_report.json
INFO:src.core.result_saver:绩效报告CSV已保存: outputs\Crack_Spread_Results\strategy_runs\20251102_180742\performance_report.csv
INFO:src.core.result_saver:交易日志已保存: outputs\Crack_Spread_Results\strategy_runs\20251102_180742\performance_trades.csv
INFO:src.core.result_saver:结果保存器初始化完成，输出目录: outputs\Crack_Spread_Results\strategy_runs\20251102_180742
INFO:src.core.result_saver:策略配置已保存: outputs\Crack_Spread_Results\strategy_runs\20251102_180742\strategy_c


[8/9] 保存策略结果...
  [7.1] 保存策略配置...
    ✓ 策略配置已保存
  [7.2] 保存绩效报告...
    ✓ 绩效报告已保存:
      - report_json: performance_report.json
      - report_csv: performance_report.csv
      - trades: performance_trades.csv
      - equity_curve: performance_equity_curve.csv
      - summary_txt: performance_summary.txt
  [7.3] 保存特征信息...
    ✓ 特征信息已保存
  [7.4] 保存模型指标...
    ✓ 模型指标已保存
      训练集准确率: 0.0000
      测试集准确率: 0.0000
  [7.5] 保存交易信号...
    ✓ 交易信号已保存
  [7.6] 创建README...
    ✓ README已创建

✓ 所有结果已保存到: outputs\Crack_Spread_Results\strategy_runs\20251102_180742

📂 生成的文件:
  📄 feature_importance.csv              (    1.72 KB)
  📄 feature_info.json                   (    6.73 KB)
  📄 model_metrics.json                  (    0.39 KB)
  📄 performance_equity_curve.csv        (  101.43 KB)
  📄 performance_report.csv              (    0.41 KB)
  📄 performance_report.json             (    0.50 KB)
  📄 performance_summary.txt             (    1.57 KB)
  📄 performance_trades.csv              (   46.54 KB)
  📄 REA

# 超参数优化

使用不同的搜索方法优化模型超参数：
- 网格搜索（Grid Search）：遍历所有参数组合
- 随机搜索（Random Search）：随机采样参数组合
- 贝叶斯优化（Bayesian Optimization）：智能搜索最优参数

## 步骤：
1. 导入超参数优化模块
2. 选择搜索方法
3. 执行参数搜索
4. 比较不同方法的结果
5. 使用最佳参数重新训练模型

In [12]:
# 导入超参数优化模块
from src.core.hyperparameter_tuning import HyperparameterTuner

# 创建超参数优化器
tuner = HyperparameterTuner(
    model_type='gradient_boosting',
    task='classification',
    scoring='f1_weighted',  # 使用加权F1分数
    cv=5,  # 5折交叉验证
    n_jobs=-1,  # 使用所有CPU核心
    verbose=1
)

print("超参数优化器初始化完成")
print(f"模型类型: {tuner.model_type}")
print(f"评分指标: {tuner.scoring}")
print(f"交叉验证折数: {tuner.cv}")

超参数优化器初始化完成
模型类型: gradient_boosting
评分指标: f1_weighted
交叉验证折数: 5


## 方法1：网格搜索（Grid Search）

网格搜索会遍历所有可能的参数组合，找到最优参数。

**优点**：能找到全局最优解（在给定的参数空间内）  
**缺点**：计算成本高，参数组合数呈指数增长  
**适用场景**：参数空间较小，计算资源充足

In [13]:
# # 方法1：网格搜索
# print("\n" + "="*60)
# print("方法1：网格搜索（Grid Search）")
# print("="*60)

# # 创建基础模型
# from sklearn.ensemble import GradientBoostingClassifier
# from src.core.hyperparameter_tuning import HyperparameterTuner
# base_model_grid = GradientBoostingClassifier(random_state=42)

# tuner=HyperparameterTuner(
#     model_type='gradient_boosting',
#     task='classification',
#     cv=5,)
# # 执行网格搜索
# best_params_grid = tuner.grid_search(
#     model=base_model_grid,
#     X_train=X_train,
#     y_train=y_train
# )

# print("\n网格搜索结果:")
# print(f"最佳参数: {best_params_grid}")
# print(f"最佳得分: {tuner.best_score_:.4f}")

# # 显示前10个最佳参数组合
# print("\n前10个最佳参数组合:")
# summary_grid = tuner.get_search_results_summary()
# print(summary_grid.head(10).to_string())

## 使用最佳参数重新训练模型

使用搜索到的最佳参数重新训练模型，并评估性能提升。

In [14]:
# # 使用最佳参数重新训练模型
# print("\n" + "="*60)
# print("使用最佳参数重新训练模型")
# print("="*60)
# best_params_final = best_params_grid
# best_tuner = tuner
# # 选择最佳方法的参数
# # if best_method == 'Grid Search':
# #     best_params_final = best_params_grid
# #     best_tuner = tuner
# # elif best_method == 'Random Search':
# #     best_params_final = best_params_random
# #     best_tuner = tuner_random
# # else:
# #     best_params_final = best_params_bayes
# #     best_tuner = tuner_bayes

# # print(f"\n使用 {best_method} 的最佳参数:")
# for param, value in best_params_final.items():
#     print(f"  {param}: {value}")

# # 使用最佳参数创建新模型
# model_optimized = MLModel(model_type='gradient_boosting', task='classification')

# # 使用最佳参数训练
# print("\n训练优化后的模型...")
# model_optimized.model = GradientBoostingClassifier(**best_params_final, random_state=42)
# model_optimized.model.fit(X_train, y_train)

# # 评估优化后的模型
# metrics_optimized = model_optimized.evaluate(X_test, y_test)

# print("\n优化后模型性能:")
# print(f"  准确率: {metrics_optimized.get('accuracy', 0):.4f}")
# print(f"  F1分数: {metrics_optimized.get('f1', 0):.4f}")
# print(f"  精确率: {metrics_optimized.get('precision', 0):.4f}")
# print(f"  召回率: {metrics_optimized.get('recall', 0):.4f}")

# # 与原始模型比较
# print("\n性能对比:")
# print(f"  原始模型 F1: {metrics.get('f1', 0):.4f}")
# print(f"  优化模型 F1: {metrics_optimized.get('f1', 0):.4f}")
# improvement = (metrics_optimized.get('f1', 0) - metrics.get('f1', 0)) / metrics.get('f1', 1) * 100
# print(f"  提升幅度: {improvement:+.2f}%")

# # 保存优化结果
# best_tuner.save_results('models/tuning_results')
# print("\n✓ 优化结果已保存到 models/tuning_results/")

# # 更新全局model变量为优化后的模型
# model = model_optimized
# print("\n✓ 全局模型已更新为优化后的模型")

In [15]:
signal_generator = SignalGenerator(model, threshold=0.5, signal_holding_days=20)
signals = signal_generator.generate_signals(X_test, use_probability=True)
signals.index = test_idx
print(f"    ✓ 生成 {len(signals)} 个信号")
print(f"    信号分布: {signals.value_counts().to_dict()}")

# 获取价差价格数据
print("  [5.2] 准备价格数据...")
# 确保spread_data索引与test_idx时区一致
spread_df_for_backtest = spread_data['CRACK_3_2_1'].copy()
if hasattr(spread_df_for_backtest.index, 'tz') and spread_df_for_backtest.index.tz is not None:
    spread_df_for_backtest.index = spread_df_for_backtest.index.tz_localize(None)

spread_prices = spread_df_for_backtest.loc[test_idx, ['spread']].copy()
spread_prices.columns = ['close']
spread_prices['volatility'] = spread_prices['close'].pct_change().rolling(20).std()
print(f"    ✓ 价格数据: {len(spread_prices)} 条")

# 运行回测
print("  [5.3] 运行回测...")
backtest_engine = BacktestEngine(
    initial_capital=1000000,
    commission_rate=0.0005,
    slippage_rate=0.0001,
    max_position=1e16,
    max_capital_usage=0.05
)

equity_curve = backtest_engine.run_backtest(
    spread_prices,
    signals,
    price_col='close',
    volatility_col='volatility'
)
print(f"    ✓ 回测完成，最终权益: ${equity_curve['equity'].iloc[-1]:,.2f}")

# 获取交易日志
trade_log = backtest_engine.get_trade_log()
print(f"    ✓ 总交易次数: {len(trade_log)}")

# 绩效分析
print("  [5.4] 绩效分析...")
analyzer = PerformanceAnalyzer(
    equity_curve,
    initial_capital=1000000,
    risk_free_rate=0.02
)

performance_report = analyzer.generate_performance_report(trade_log)
print(f"    ✓ 总收益率: {performance_report.get('total_return', 0)*100:.2f}%")
print(f"    ✓ 夏普比率: {performance_report.get('sharpe_ratio', 0):.2f}")
print(f"    ✓ 最大回撤: {performance_report.get('max_drawdown', 0)*100:.2f}%")

print("✓ 回测完成")
logger.info("回测完成\n")

# 🔍 调试点6：在此处设置断点，检查回测结果
# 可以查看: equity_curve.tail(), trade_log.head(), performance_report

# ============================================================
# 步骤6：可视化
# ============================================================
print("\n[7/9] 开始可视化...")
logger.info("\n" + "="*60)
logger.info("步骤6：结果可视化")
logger.info("="*60)

print("  [6.1] 生成价格和价差图...")
visualizer.plot_price_and_spread(
    price_data,
    spread_data['CRACK_3_2_1'],
    title='Crack Spread 3:2:1'
)
print("    ✓ price_spread_chart.png")

print("  [6.2] 生成权益曲线图...")
visualizer.plot_equity_curve(equity_curve)
print("    ✓ equity_curve.png")

print("  [6.3] 生成收益率分布图...")
returns = equity_curve['equity'].pct_change().dropna()
visualizer.plot_returns_distribution(returns)
print("    ✓ returns_distribution.png")

print("  [6.4] 生成月度收益热力图...")
visualizer.plot_monthly_returns_heatmap(equity_curve)
print("    ✓ monthly_returns_heatmap.png")

print("  [6.5] 生成滚动指标图...")
visualizer.plot_rolling_metrics(equity_curve, window=60)
print("    ✓ rolling_metrics.png")

print("  [6.6] 生成交易分析图...")
visualizer.plot_trade_analysis(trade_log)
print("    ✓ trade_analysis.png")

print("✓ 可视化完成")
logger.info("可视化完成\n")

INFO:src.core.ml_models:使用固定阈值模式: 上阈值=0.05, 下阈值=-0.05
INFO:src.core.ml_models:回归信号统计:
INFO:src.core.ml_models:  预测值范围: [-9.5793, 3.0636]
INFO:src.core.ml_models:  做多信号(1): 195 (20.9%)
INFO:src.core.ml_models:  观望信号(0): 26 (2.8%)


INFO:src.core.ml_models:  做空信号(-1): 711 (76.3%)
INFO:src.core.ml_models:生成交易信号完成，信号分布:
INFO:src.core.ml_models:-1    711
 1    195
 0     26
Name: signal, dtype: int64
INFO:src.core.ml_models:应用20天信号维持后，信号分布:
INFO:src.core.ml_models:-1    731
 1    201
Name: signal, dtype: int64


    ✓ 生成 932 个信号
    信号分布: {(-9.579294184083587, -1, 1.0): 1, (-0.3533807406547973, -1, 1.0): 1, (-0.3794884997541652, -1, 1.0): 1, (-0.3792486093420331, -1, 1.0): 1, (-0.37306102171412014, -1, 1.0): 1, (-0.37295935528537677, -1, 1.0): 1, (-0.3659872277123976, -1, 1.0): 1, (-0.3638887545527088, -1, 1.0): 1, (-0.3617917431521986, -1, 1.0): 1, (-0.360657887313367, -1, 1.0): 1, (-0.3595096008623464, -1, 1.0): 1, (-0.3589449011858663, -1, 1.0): 1, (-0.35791861404727077, -1, 1.0): 1, (-0.3578565365048274, -1, 1.0): 1, (-0.34881432632859416, -1, 1.0): 1, (-0.4732404051141878, -1, 1.0): 1, (-0.3476050941622225, -1, 1.0): 1, (-0.34580711459260105, -1, 1.0): 1, (-0.34180902670569624, -1, 1.0): 1, (-0.334779758268477, -1, 1.0): 1, (-0.3330340022615893, -1, 1.0): 1, (-0.33282257783902486, -1, 1.0): 1, (-0.32847576189163796, -1, 1.0): 1, (-0.3270432594067775, -1, 1.0): 1, (-0.3251248135870263, -1, 1.0): 1, (-0.32098833133865545, -1, 1.0): 1, (-0.31729198002146547, -1, 1.0): 1, (-0.3039351806575386

KeyError: "None of [DatetimeIndex(['2022-02-03 05:00:00', '2022-02-04 05:00:00',\n               '2022-02-07 05:00:00', '2022-02-08 05:00:00',\n               '2022-02-09 05:00:00', '2022-02-10 05:00:00',\n               '2022-02-11 05:00:00', '2022-02-14 05:00:00',\n               '2022-02-15 05:00:00', '2022-02-16 05:00:00',\n               ...\n               '2025-10-06 04:00:00', '2025-10-07 04:00:00',\n               '2025-10-08 04:00:00', '2025-10-09 04:00:00',\n               '2025-10-10 04:00:00', '2025-10-13 04:00:00',\n               '2025-10-14 04:00:00', '2025-10-15 04:00:00',\n               '2025-10-16 04:00:00', '2025-10-17 04:00:00'],\n              dtype='datetime64[ns]', name='date', length=932, freq=None)] are in the [index]"